In [ ]:
!git clone https://github.com/Naungth/BluPRINT.git

fatal: destination path 'BluPRINT' already exists and is not an empty directory.


In [ ]:
!pip install pytorch_lightning
!pip install open_clip_torch
!pip install xformers

In [ ]:
import os
import shutil

base_dir = '/content/BluPRINT/dataset'
folders_to_empty = ['architectural-styles-dataset', 'g-images-dataset']

image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')
moved_count = 0

for folder_name in folders_to_empty:
    folder_path = os.path.join(base_dir, folder_name)
    if not os.path.exists(folder_path):
        print(f"Directory {folder_path} not found. Skipping.")
        continue

    # Walk through all subdirectories and files
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(image_extensions):
                source_path = os.path.join(root, file)
                destination_path = os.path.join(base_dir, file)

                shutil.move(source_path, destination_path)
                moved_count += 1

print(f"Successfully moved {moved_count} images to {base_dir}")

base_dir = '/content/BluPRINT/dataset'
folders_to_delete = ['architectural-styles-dataset', 'g-images-dataset']

for folder_name in folders_to_delete:
    folder_path = os.path.join(base_dir, folder_name)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Successfully deleted {folder_path}")
    else:
        print(f"Directory {folder_path} does not exist.")

Directory /content/BluPRINT/dataset/architectural-styles-dataset not found. Skipping.
Directory /content/BluPRINT/dataset/g-images-dataset not found. Skipping.
Successfully moved 0 images to /content/BluPRINT/dataset
Directory /content/BluPRINT/dataset/architectural-styles-dataset does not exist.
Directory /content/BluPRINT/dataset/g-images-dataset does not exist.


In [ ]:
import json
import cv2
import numpy as np
import os

from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self):
        self.data = []
        with open('/content/BluPRINT/training_data_qwen.jsonl', 'rt') as f:
            for line in f:
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        source_filename = os.path.basename(item['target'])
        target_filename = os.path.basename(item['source'])
        prompt = item['prompt']

        # Use os.path.join to safely construct the file paths
        source = cv2.imread(os.path.join('/content/BluPRINT/dataset', target_filename))
        target = cv2.imread(os.path.join('/content/BluPRINT/hint_images', source_filename))

        # If the image wasn't found, raise a clear error rather than failing in cvtColor
        if source is None:
            raise FileNotFoundError(f"Source image not found: {os.path.join('/BluPRINT/content/dataset', target_filename)}")
        if target is None:
            raise FileNotFoundError(f"Target image not found: {os.path.join('/BluPRINT/content/hint_images', source_filename)}")

        # Do not forget that OpenCV read images in BGR order.
        source = cv2.cvtColor(source, cv2.COLOR_BGR2RGB)
        target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

        # Resize images to a consistent shape to prevent DataLoader batching errors
        source = cv2.resize(source, (512, 512))
        target = cv2.resize(target, (512, 512))

        # Normalize source images to [0, 1].
        source = source.astype(np.float32) / 255.0

        # Normalize target images to [-1, 1].
        target = (target.astype(np.float32) / 127.5) - 1.0

        return dict(jpg=target, txt=prompt, hint=source)


In [24]:
from torch.utils.data import random_split, DataLoader

# Load full dataset
full_dataset = MyDataset()

# Calculate split sizes (90% train, 10% validation)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

# Split the dataset randomly
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create your DataLoaders (Note: validation shouldn't be shuffled)
train_dataloader = DataLoader(train_dataset, num_workers=8, batch_size=8, shuffle=True)
val_dataloader = DataLoader(val_dataset, num_workers=8, batch_size=8, shuffle=False)

In [18]:
import torch
import torchvision
from PIL import Image
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.utilities.rank_zero import rank_zero_only


class ImageLogger(Callback):
    def __init__(self, batch_frequency=2000, max_images=4, clamp=True, increase_log_steps=True,
                 rescale=True, disabled=False, log_on_batch_idx=False, log_first_step=False,
                 log_images_kwargs=None):
        super().__init__()
        self.rescale = rescale
        self.batch_freq = batch_frequency
        self.max_images = max_images
        if not increase_log_steps:
            self.log_steps = [self.batch_freq]
        self.clamp = clamp
        self.disabled = disabled
        self.log_on_batch_idx = log_on_batch_idx
        self.log_images_kwargs = log_images_kwargs if log_images_kwargs else {}
        self.log_first_step = log_first_step

    @rank_zero_only
    def log_local(self, save_dir, split, images, global_step, current_epoch, batch_idx):
        root = os.path.join(save_dir, "image_log", split)
        for k in images:
            grid = torchvision.utils.make_grid(images[k], nrow=4)
            if self.rescale:
                grid = (grid + 1.0) / 2.0  # -1,1 -> 0,1; c,h,w
            grid = grid.transpose(0, 1).transpose(1, 2).squeeze(-1)
            grid = grid.numpy()
            grid = (grid * 255).astype(np.uint8)
            filename = "{}_gs-{:06}_e-{:06}_b-{:06}.png".format(k, global_step, current_epoch, batch_idx)
            path = os.path.join(root, filename)
            os.makedirs(os.path.split(path)[0], exist_ok=True)
            Image.fromarray(grid).save(path)

    def log_img(self, pl_module, batch, batch_idx, split="train"):
        check_idx = batch_idx  # if self.log_on_batch_idx else pl_module.global_step
        if (self.check_frequency(check_idx) and  # batch_idx % self.batch_freq == 0
                hasattr(pl_module, "log_images") and
                callable(pl_module.log_images) and
                self.max_images > 0):
            logger = type(pl_module.logger)

            is_train = pl_module.training
            if is_train:
                pl_module.eval()

            with torch.no_grad():
                images = pl_module.log_images(batch, split=split, **self.log_images_kwargs)

            for k in images:
                N = min(images[k].shape[0], self.max_images)
                images[k] = images[k][:N]
                if isinstance(images[k], torch.Tensor):
                    images[k] = images[k].detach().cpu()
                    if self.clamp:
                        images[k] = torch.clamp(images[k], -1., 1.)

            self.log_local(pl_module.logger.save_dir, split, images,
                           pl_module.global_step, pl_module.current_epoch, batch_idx)

            if is_train:
                pl_module.train()

    def check_frequency(self, check_idx):
        return check_idx % self.batch_freq == 0

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):
        if not self.disabled:
            self.log_img(pl_module, batch, batch_idx, split="train")

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):
        if not self.disabled:
            self.log_img(pl_module, batch, batch_idx, split="val")


In [ ]:
import importlib

def get_obj_from_str(string, reload=False):
    module, cls = string.rsplit(".", 1)
    if reload:
        module_imp = importlib.import_module(module)
        importlib.reload(module_imp)
    return getattr(importlib.import_module(module, package=None), cls)

def instantiate_from_config(config):
    if not "target" in config:
        if config == '__is_first_stage__':
            return None
        elif config == "__is_unconditional__":
            return None
        raise KeyError("Expected key `target` to instantiate.")
    return get_obj_from_str(config["target"])(**config.get("params", dict()))

In [ ]:
!sed -i 's/pytorch_lightning.utilities.distributed/pytorch_lightning.utilities.rank_zero/g' /content/BluPRINT/ControlNet/ldm/models/diffusion/ddpm.py

In [ ]:
!sed -i 's/def on_train_batch_start(self, batch, batch_idx, dataloader_idx):/def on_train_batch_start(self, batch, batch_idx, dataloader_idx=0):/g' /content/BluPRINT/ControlNet/ldm/models/diffusion/ddpm.py

In [ ]:
from omegaconf import OmegaConf

def get_state_dict(d):
    return d.get('state_dict', d)

def load_state_dict(ckpt_path, location='cpu'):
    _, extension = os.path.splitext(ckpt_path)
    if extension.lower() == ".safetensors":
        import safetensors.torch
        state_dict = safetensors.torch.load_file(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    state_dict = get_state_dict(state_dict)
    print(f'Loaded state_dict from [{ckpt_path}]')
    return state_dict

def create_model(config_path):
    config = OmegaConf.load(config_path)
    model = instantiate_from_config(config.model).cpu()
    print(f'Loaded model config from [{config_path}]')
    return model

In [ ]:
resume_path = '/content/BluPRINT/ControlNet/models/control_sd15_ini.ckpt'
if not os.path.exists(resume_path):
    print("Downloading initial checkpoint...")
    os.system(f"wget -q -O {resume_path} https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt")

In [ ]:
import sys
import os
import subprocess
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import TQDMProgressBar

# Add ControlNet to sys.path so 'cldm' and other modules can be found
sys.path.append('/content/BluPRINT/ControlNet')

from torch.utils.data import DataLoader

# Enable Tensor Cores (TF32) for better performance on Ampere+ GPUs like the A100
torch.set_float32_matmul_precision('medium')

# Configs
resume_path = '/content/BluPRINT/ControlNet/models/control_sd15_ini.ckpt'
prepared_path = '/content/BluPRINT/ControlNet/models/control_sd15_prepared.ckpt'
batch_size = 8
logger_freq = 300
learning_rate = 1e-5
sd_locked = True
only_mid_control = False

# Fix PyTorch 2.6 weights_only=True issue in tool_add_control.py
os.system("sed -i 's/torch.load(input_path)/torch.load(input_path, weights_only=False)/g' /content/BluPRINT/ControlNet/tool_add_control.py")

# Prepare the ControlNet initial checkpoint if not already prepared
if not os.path.exists(prepared_path):
    print("Adding ControlNet parameters to the base model...")
    try:
        result = subprocess.run(
            ['python', 'tool_add_control.py', resume_path, prepared_path],
            cwd='/content/BluPRINT/ControlNet',
            check=True,
            capture_output=True,
            text=True
        )
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("Error running tool_add_control.py:")
        print(e.stdout)
        print(e.stderr)
        raise e

# First use cpu to load models. Pytorch Lightning will automatically move it to GPUs.
model = create_model('/content/BluPRINT/ControlNet/models/cldm_v15.yaml').cpu()

# Load the state dict using weights_only=False to bypass PyTorch 2.6 security restrictions
state_dict = torch.load(prepared_path, map_location='cpu', weights_only=False)
if 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']
model.load_state_dict(state_dict)

model.learning_rate = learning_rate
model.sd_locked = sd_locked
model.only_mid_control = only_mid_control

# Misc
dataset = MyDataset()
dataloader = DataLoader(dataset, num_workers=8, batch_size=batch_size, shuffle=True)
logger = ImageLogger(batch_frequency=logger_freq)
trainer = pl.Trainer(accelerator='gpu', devices=1, precision=32, callbacks=[logger, TQDMProgressBar()], max_epochs=250)

os.chdir('/content/BluPRINT/ControlNet')

# Train!
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)


ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}

Loaded model config from [/content/BluPRINT/ControlNet/models/cldm_v15.yaml]


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Data shape for DDIM sampling is (4, 4, 64, 64), eta 0.0
Running DDIM Sampling with 50 timesteps



DDIM Sampler: 100%|██████████| 50/50 [00:12<00:00,  3.95it/s]


Training: |          | 0/? [00:00<?, ?it/s]

Data shape for DDIM sampling is (4, 4, 64, 64), eta 0.0
Running DDIM Sampling with 50 timesteps



DDIM Sampler: 100%|██████████| 50/50 [00:12<00:00,  3.93it/s]
